In [ ]:
%pip install --upgrade pip setuptools wheel


In [ ]:
%pip install -U \
    category-encoders \
    # cffi==1.16.0 \
    cloudpickle==3.0.0 \
    nltk==3.9.2 \
    defusedxml==0.7.1 \
    graphviz==0.20.3 \
    holidays==0.54 \
    lightgbm==4.5.0 \
    # lz4==4.3.3 \
    matplotlib==3.9.2 \
    psutil==5.9.8 \
    pyarrow==15.0.2 \
    optuna


In [ ]:
%pip install optuna 

In [ ]:
%pip install mlflow==2.15.1

In [ ]:
import mlflow

In [ ]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [ ]:
import pandas as pd
train = pd.read_csv(
    "/Users/saurabh.prajapati/Documents/Medisyn-Labs/data/train.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)
train['positive'] = (train['rating'] > 6).astype(int)

In [ ]:
import pandas as pd
test = pd.read_csv(
    "/Users/saurabh.prajapati/Documents/Medisyn-Labs/data/test.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)
test['positive'] = (test['rating'] > 6).astype(int)

In [ ]:
df=pd.concat([train,test],axis=0)
df.drop(['rating','customer_identifier'],axis=1,inplace=True)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
train.shape

In [ ]:
target_col='positive'
nlp_col=['review_benefits','review_sideEffects','review_overall']
boolean_cols = df.select_dtypes(include="boolean").columns
numerical_cols = df.select_dtypes(include="number").columns
numerical_cols = list(set(numerical_cols)-set([target_col]))
categorical_cols = df.select_dtypes(include="object").columns
categorical_cols=list(set(categorical_cols)-set(nlp_col))

summary = {
    "boolean_cols": boolean_cols,
    "numerical_cols": numerical_cols,
    "categorical_cols": categorical_cols
}

for col_type, cols in summary.items():
    print(f"{col_type} ({len(cols)} columns):")
    for col in cols:
        print(f" - {col}")
    print()

In [ ]:
cardinality = {"low": [], "medium": [], "high": []}

for col in categorical_cols:
    unique_count = df[col].nunique()
    if unique_count < 10:
        cardinality["low"].append((col, unique_count))
    elif 10 <= unique_count <= 50:
        cardinality["medium"].append((col, unique_count))
    else:
        cardinality["high"].append((col, unique_count))

for key, value in cardinality.items():
    print(f"{key.capitalize()} cardinality ({len(value)} columns):")
    for col, count in value:
        print(f" - {col}: {count}")
    print()
    
low_cardinal_cat_feats = [col for col, _ in cardinality['low']]
meidum_cardinal_cat_feats = [col for col, _ in cardinality['medium']]
high_cardinal_cat_feats = [col for col, _ in cardinality['high']]

In [ ]:
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context


In [ ]:
import numpy as np
import pandas as pd
import re, emoji, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm
from category_encoders import CatBoostEncoder, TargetEncoder
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ----------------------------------------------------------------
# 🔹 1. Download NLTK resources (one-time)
# ----------------------------------------------------------------

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

# ----------------------------------------------------------------
# 🔹 2. Stopwords and Lemmatizer setup
# ----------------------------------------------------------------
additional_stopwords = {
    "drug", "medicine", "tablet", "doctor", "patient", "review_benefits", "review_sideEffects",
    "review_overall", "mg", "day", "take", "took", "used", "taking", "pill", "one", "review",
    "dose", "dosage", "medication", "treatment", "therapy", "prescribed", "prescription", 
    "prescribe", "physician", "med", "antibiotic", "cream", "application", "apply", "use", 
    "using", "take", "taken", "stop", "stopped", "started", "starting", "start", "continue",
    "continued", "course", "treat", "treated", "treating", "dos", "mcg", "tab", "daily", 
    "nightly", "bedtime", "morning", "evening", "hour", "per", "pm", "qd", "weekly", "month",
    "week", "year", "night", "time", "two", "three", "four", "five", "every", "twice", "long",
    "within", "since", "around", "last", "next", "ago", "short",
    "january", "february", "march", "april", "may", "june", "july", "august", "september",
    "october", "november", "december", "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep",
    "sept", "oct", "nov", "dec"
}

stop_words = set(stopwords.words('english')).union(additional_stopwords)
lemmatizer = WordNetLemmatizer()

# ----------------------------------------------------------------
# 🔹 3. Text Preprocessor Function
# ----------------------------------------------------------------
def preprocess_text_fast(text):
    if not isinstance(text, str):
        return ""
    text = emoji.demojize(text, language='en')
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

def nlp_text(X):
    X = X.copy()
    for col in X.columns:
        X[col] = X[col].apply(preprocess_text_fast)
    return X

# Updated cleaner to return Series
def nlp_text_series(x):
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]  # Take first (only) column
    return x.apply(preprocess_text_fast)

# ----------------------------------------------------------------
# 🔹 4. Feature Groups
# ----------------------------------------------------------------

# ----------------------------------------------------------------
# 🔹 5. OneHotEncoder Pipeline (low-cardinality)
# ----------------------------------------------------------------
one_hot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
     ("scaler", StandardScaler())
])
CatBoostEncoder._get_tags = lambda self: {"allow_nan": True}
# ----------------------------------------------------------------
# 🔹 6. CatBoostEncoder Pipeline (high-cardinality)
# ----------------------------------------------------------------
catboost_pipeline = Pipeline([
    ("imputer", SimpleImputer(fill_value="NOT_AVAILABLE", strategy="constant")),
    ("encoder", CatBoostEncoder()),
    ("scaler", StandardScaler())
])

# ----------------------------------------------------------------
# 🔹 7. NLP Transformer (text column)
# ----------------------------------------------------------------
nlp_pipeline = Pipeline([
    ("cleaner", FunctionTransformer(nlp_text_series, validate=False)),
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2)))
])

# ----------------------------------------------------------------
# 🔹 8. Combine All Transformers
# ----------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("one_hot", one_hot_pipeline, low_cardinal_cat_feats),
        ("catboost", catboost_pipeline, high_cardinal_cat_feats),
        ("nlp", nlp_pipeline, nlp_col),
    ],
    remainder="passthrough",
    sparse_threshold=1
)


In [ ]:
preprocessor

In [ ]:
req_cols=["medicine_name","effectiveness","side_effects","illness"]+ nlp_col

X_train, X_test, y_train, y_test = train[req_cols], test[req_cols],train[target_col],test[target_col]

In [ ]:
summary = {
    "name": ["Train Set","Test Set"],
    "# samples": [X_train.shape[0], X_test.shape[0]],
    "positive_rate": [
        round(100 * y_train.sum() / y_train.shape[0], 2),
        round(100 * y_test.sum() / y_test.shape[0], 2)
    ]
}
pd.DataFrame(summary)

In [ ]:
import optuna

In [ ]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_validate, StratifiedKFold
from lightgbm import LGBMClassifier
import optuna

# Define the objective function for Optuna
def objective(trial):
    param = {
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.1, 0.5),
        "max_bin": trial.suggest_int("max_bin", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 6, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "n_estimators": trial.suggest_int("n_estimators", 5, 100),
        "num_leaves": trial.suggest_int("num_leaves", 20, 50),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 100.0),
        "random_state": 537672287,
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
    }

    model = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", LGBMClassifier(**param)),
        ]
    )

    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        return_train_score=True,
        n_jobs=-1,
    )
    return cv_results["test_score"].mean()

# Create a study and optimize the objective function
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Get the best parameters
best_params = study.best_params
best_params["random_state"] = 537672287


model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(**best_params)),
    ]
)

# Fit the model
model.fit(X_train, y_train)

# Stratified K-Fold for handling imbalanced classes
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)

# Evaluate model with cross-validation
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    return_train_score=True,
    n_jobs=-1
)

In [91]:
from helper import *

best_proba_threshold = get_proba_threshold(model, X_test, y_test)
path="/Users/saurabh.prajapati/Documents/Medisyn-Labs/data/evaluation_metrics/eval_metrics_sentiment_model.csv"

eval_metrics=log_model_eval_metrics(model, X_train, y_train, X_test, y_test, best_proba_threshold)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=7.512277003698922, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.512277003698922
[LightGBM] [Warning] lambda_l2 is set=7.979487721678669, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.979487721678669
[LightGBM] [Warning] lambda_l1 is set=7.512277003698922, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.512277003698922
[LightGBM] [Warning] lambda_l2 is set=7.979487721678669, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.979487721678669
[LightGBM] [Warning] lambda_l1 is set=7.512277003698922, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.512277003698922
[LightGBM] [Warning] lambda_l2 is set=7.979487721678669, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.979487721678669
{'f1_score': [0.93, 0.91], 'precision': [0.9, 0.89], 'recall': [0.96, 0.94], 'roc_auc': [0.96, 0.95]}


/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [92]:
best_proba_threshold

0.458

In [102]:
eval_metrics 
path="/Users/saurabh.prajapati/Documents/Medisyn-Labs/data/evaluation_metrics/eval_metrics_sentiment.csv"
eval_metrics.to_csv(path,index=False)

In [105]:
import numpy as np

apply_model_and_get_ks_table(model, X_train, y_train, X_test, y_test,  best_proba_threshold, y_true_col='positive', y_pred_proba_col='proba', verbose=True)


[LightGBM] [Warning] lambda_l1 is set=7.512277003698922, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.512277003698922
[LightGBM] [Warning] lambda_l2 is set=7.979487721678669, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.979487721678669
[LightGBM] [Warning] lambda_l1 is set=7.512277003698922, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7.512277003698922
[LightGBM] [Warning] lambda_l2 is set=7.979487721678669, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.979487721678669
KS is 77.93% at Decile 7
KS is 71.29% at Decile 6


/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
